In [ ]:
# Relevant python functions
import pandas as pd
import numpy as np
import geopandas as gpd
import os
import sys
from pyproj import Transformer
from shapely.geometry import Point, box
import matplotlib.pyplot as plt
import folium
import contextily as ctx
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from shapely import wkt

# BRAILS functions for imputation 
from brails.types.image_set import ImageSet    
from brails.types.asset_inventory import Asset, AssetInventory
from brails.utils import Importer

# Import functions for inventory generation 
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
fxn_dir = os.path.join(parent_dir, "inventory_generation_functions")
sys.path.append(fxn_dir)

import functions_general as fxns
import functions_preprocessing as pre
import functions_point_to_ftpt as pt_ftpt
import functions_disagreement_and_gaps as resolve


# Set plotting CRS values for data manipulation and plotting
crs_main = '26910' # Used for data manipulation and storage
crs_plot = '4269' # Used for plotting 

# HAYWARD BOUNDS FOR PLOTTING 
xbounds = [-122.15, -122.02]
ybounds = [37.60, 37.69]

# Set state, state fips, and county for download
state = 'California'
state_fips = '06'
county = 'Alameda'
county_fips = '001'

# Load API Key
with open("./Input_Data/Census/census_api_key.txt", "r", encoding="utf-8") as file:
    census_api_key = file.read()

In [ ]:
## Set name of inventory for final output 
name = 'Synthesized_National_SampleYear'

In [ ]:
## FLAGS THAT CONTROL FOOTPRINTS AND SIZE FILTERING 

## Select Footprint source and size limit - can only select values already used to generate preprocessed data
footprint_name = 'Hayward_Footprints' 

########### Sqft limit below which to filter small NSI POINTS ##########
nsi_filter_limit = 450 # options are 0, 450, 800

########### Set flag to determine how footprints are designated as "partially full" ##########
estimate_stories = True

########### Options to prioritize partial and/or use full footprints  ##########
use_full_ftpt = True
prioritize_partial_footprints = True

########### How far away to we let data travel (conducted after 10m merge) ##########
final_distance_bound = 100

########### Which cost to prioritize ##########
cost_source = 'nsi' # options: 'hazus', 'nsi'

########### Unit source ##########
unit_source = 'census' # nsi_mean, nsi_min, nsi_max, census 

########### Area source ##########
area_source = 'nsi' # options: 'footprint', 'nsi'

# Use building type (material) to constrain list of possible structure types  
use_nsi_bldg_type = False

# Set flag to determine whether to sample year built values from a distribution, or use the raw values from NSI26
sample_year = True


In [ ]:
## Set flag to include HIFLD data or not 
include_hifld = True

########### Flag to keep or drop NSI EDU1 and GOV2 points not nearby HIFLD data
drop_unpaired_nsi_edu1 = False # Flag to drop all NSI points, other than those closest to HIFLD school points (max 50 m) 
drop_edu1_far_from_hifld = True # Flag to drop all NSI EDU1 points farther than a certain radius from NSI 
gov1_near_edu1 = 'convert' # options: drop, convert (converts to EDU1), or keep 

drop_unpaired_nsi_gov2 = False # Flag to drop all NSI points, other than those closest to HIFLD GOV2 points (max 50 m)
drop_gov1_near_gov2 = True # Flag to drop all NSI GOV1 points within 10m
gov2_far_from_hifld = 'convert'  # options: drop, convert (converts to GOV1), or keep 


########### 4) Set flag to determine if NSI population should be scaled within campus polygons to match HIFLD campus enrollment ##########
scale_edu2_pop = True

In [ ]:
## FLAGS THAT CONTROL HOW MERGE IS DONE 

########### Set flag to drop GOV1, IND5, and IND5 if they do not intersect with footprints ##########
drop_gov1_ind4_ind5 = True

########### 8) For unpaired NSI points, should footprints in the surrounding census blocks be considered as potential pairings? ##########
use_surrounding_bgs = True

In [ ]:
## FLAGS TO CONTROL MOBILE HOMES AND STRUCTURE TYPE 

## Set flag to include manual MH polygons
include_manual_mh_polygons = True

# Allows structure type 'MH' only when the occupancy class is RES2 (diverges from Hazus) 
allow_mh_only_for_res2 = True 

# Doesn't allow the assignment of URM buildings, due to efforts to retrofit those buildings (diverges from Hazus)
no_urm = True

# Adopts structure types used for 'RES1' to be assigned for RES3A and RES3B. 2-4 unit structures are likely structurally more similar to single family homes than to large apartment buildings. 
res3ab_to_res1_flag = True 


# For cases where structure type doesn't exist for combintion of occupancy/height/region, fill in by sampling randomly 
fill_in_error_structure_types = True

In [ ]:
## FLAGS THAT CONTROL CONFLICT RESOLUTION / GAPS 

########### 11) Scaling stories ##########
# Set stories limit: Buildings with MORE than this number of stories will be modified 
stories_limit = 12 # Tallest building in Hayward (based on google search) is ~11 stories tall 

# 12) Select which method to use to modify the very tall structures using "reset_very_high_stories"
# Selecting "Mean_of_Occupancy_Class" will reset the number of stories in the footprints exceeding stories_limit to be the mean number of stories for the given occupancy class
# Selecting "Scale_from_Units" will set the number of stories by scaling from the best estimate of the number of units. The average unit is assumed to be 1000 square feet, so estimated stories are computed as (units * 1000) / (footprint area)
# Selecting "No_Reset" does not modify the number of stories relative to the original NSI data 
reset_very_high_stories = 'No_Reset'

In [ ]:
## LOAD FOOTPRINTS AND CENSUS BLOCKS 

# Select Footprint Source
footprints = fxns.json_to_gdf(f'./Input_Data/ProcessedFootprints/{footprint_name}.json', crs_main)
footprints['FootprintID'] = footprints['FootprintID'].astype('Int64')

# Assign total square footage used for prioritizing attribution of points to footprints using "not full footprint"
footprints = pre.estimate_ftpt_size_for_merge(footprints.copy(),estimate_stories)


# Load Census blocks and tracts (generated in preprocessing file)
hayward_blocks = gpd.read_file('./Input_Data/Census/Census2020/Hayward_blocks.geojson')
hayward_tracts = gpd.read_file('./Input_Data/Census/Census2020/Hayward_tracts.geojson')

# Create CensusBlock column 
hayward_blocks['CensusBlock'] = hayward_blocks['GEOID20'] 

## PREPROCESS NSI DATA

In [ ]:
## Create folder for preprocessed data 
directory = './Input_Data/ProcessedData/National/'
os.makedirs(directory, exist_ok=True)
dir_intermediate = directory + 'Intermediate/'
os.makedirs(dir_intermediate, exist_ok=True)

In [ ]:
##### LOAD NSI DATA DOWNLOADED USING BRAILS TOOL AND CONVERT TO APPROPRIATE CRS #####
nsi = gpd.read_file('./Input_Data/National/06001.gpkg')
nsi = nsi.to_crs(epsg=crs_main)


# ##### RENAME AND DROP NSI COLUMNS AS APPROPRIATE #####
nsi = pre.rename_nsi_data26(nsi.copy())


# # Merge NSI data with City-Specific Census Blocks and check for errors in NSI data
nsi = nsi[nsi['CensusBlock'].isin(hayward_blocks['GEOID20'].to_list())].copy()

## Merge Census Tract 
nsi = nsi.sjoin(hayward_tracts[['GEOID20','geometry']], how='left', predicate='within')
nsi = nsi.rename(columns={'GEOID20':'CensusTract'})
nsi = nsi.drop(columns = ['index_right'])

# Standardize RES1 occupancy Types for consistency with other inventories
nsi['NSI_OccupancyClass'] = nsi['NSI_OccupancyClass'].apply(lambda x: 'RES1' if 'RES1' in str(x) else x)

##### CREATE ADDITIONAL COLUMNS TO BE USED IN FOOTPRINT MERGE #####
nsi['NSI_OC_Update'] = None # This will contain updated Occupancy Class values throughout merge
nsi['POINT_DropFlag'] = 0 # This indicates whether a row should be dropped from the final inventory. 1 indicates yes, 0 indicates no 
nsi['POINT_DropNote'] = "" # Space for notes on the reason data points are dropped 
nsi['POINT_Source'] = 'NSI' # This tracks the original data source for each row 
nsi['POINT_DataUpdate'] = "" # Space for notes on steps throughout update 

# Save inventory
fxns.gdf_to_json(nsi, dir_intermediate + 'Hayward_NSI.json')

In [ ]:
##### FIRST, ADDRESS EDU1 POINTS, WHICH CONSIST OF K-12 SCHOOLS #####

if include_hifld:

    ##### LOAD IMPORTED EDUCATIONAL HIFLD DATA AND CONVERT COORDINATES #####
    public = gpd.read_file('./Input_Data/HIFLD/Public_Schools.geojson')
    private = gpd.read_file('./Input_Data/HIFLD/Private_Schools.geojson')
    public = public.to_crs(epsg=crs_main)
    private = private.to_crs(epsg=crs_main)



    ##### MANUALLY REASSIGN SCHOOLS THAT WERE INCOREECTLY LOCATED - did this manually for one case in Hayward  #####
    # Ruus Elementary School 
    transformer = Transformer.from_crs("EPSG:4326", f"EPSG:{crs_main}")
    new_point = Point(transformer.transform(37.62699979615092,-122.07455518939621)) # Manually taken from Google Maps
    row_index = public[public['NAME'].str.contains('RUUS')].index[0]
    public.at[row_index, 'geometry'] = new_point



    ##### SORT FOR SCHOOLS WITHIN HAYWARD AND ASSIGN CENSUS TRACT AND BLOCK INFOMRATION #####
    # Assign blocks and clean data
    public_hayward, private_hayward, school_import = pre.format_and_locate_edu1(public, private, hayward_tracts, hayward_blocks, 'GEOID20')
    school_import = school_import.drop(columns=['NAME'])


     ##### LOAD NSI DATA #####
    nsi = fxns.json_to_gdf(dir_intermediate + 'Hayward_NSI.json', crs_main)

    # Synthesize NSI EDU1 data that is within a 50m radius of a HIFLD EDU point, drop all other NSI EDU1 points 
    # Set flag to true to display HIFLD and NSI EDU Points
    nsi = pre.synthesize_edu1_and_HIFLD_nsi26_update(nsi, school_import, drop_unpaired_nsi_edu1, drop_edu1_far_from_hifld, gov1_near_edu1)



In [ ]:
##### NEXT EDU2 POINTS, WHICH CONSIST OF COLLEGES/UNIVERSITIES #####

if include_hifld:

    ##### LOAD IMPORTED COLLEGE/UNIVERSITY HIFLD DATA AND CONVERT COORDINATES #####
    univ = gpd.read_file('./Input_Data/HIFLD/Colleges_and_Universities_Campuses.geojson')
    univ_pts = gpd.read_file('./Input_Data/HIFLD/Colleges_and_Universities.geojson')
    univ = univ.to_crs(epsg=crs_main)
    univ_pts = univ_pts.to_crs(epsg=crs_main)


    ##### SORT FOR SCHOOLS WITHIN HAYWARD AND ASSIGN CENSUS TRACT AND BLOCK INFOMRATION #####

    # Assign blocks and clean data
    univ_hayward, univ_pts_hayward = pre.locate_edu2(univ, univ_pts, hayward_tracts, hayward_blocks, 'GEOID20')

   ##### POINTS WITHOUT CAMPUS POLYGONS #####
    # Find points without associated campus polygons and append them in NSI data
    edu2_no_campus = pre.prepare_pts_without_campuses(univ_pts_hayward, univ_hayward, nsi.copy())
    nsi = pd.concat([nsi,edu2_no_campus])

    # ##### POINTS WITH CAMPUS POLYGONS, BUT WITHOUT GOV1 POINTS INSIDE #####
    # # Find points and append them in NSI data
    edu2_no_gov1 = pre.prepare_pts_without_gov1_nsi26_update(univ_pts_hayward, univ_hayward, nsi.copy())
    nsi = pd.concat([nsi,edu2_no_gov1])

    ##### POINTS WITH CAMPUS POLYGONS, WITH GOV1 POINTS INSIDE #####
    # Find GOV1 points within campus polygons and reassign them as EDU2

    # Set flag to determine if NSI population should be scaled within campus polygons to match HIFLD campus enrollment 
    scale_edu2_pop = True
    nsi = pre.merge_pts_with_campuses_nsi26_update(univ_hayward, nsi.copy(), scale_edu2_pop)

    ##### SAVE NSI #####
    # fxns.gdf_to_json(nsi, dir_intermediate + 'NSI_EDU1_EDU2_Upgrade.json')



In [ ]:
##### ADDRESS GOV2 POINTS, WHICH CONSIST OF EMERGENCY RESPONSE BUILDINGS #####

if include_hifld:


    ##### LOAD IMPORTED HIFLD DATA, CONVERT COORDINATES, AND ASSIGN CENSUS #####
    fire = gpd.read_file('./Input_Data/HIFLD/Fire_and_Emergency_Medical_Service_(EMS)_Stations.geojson')
    police = gpd.read_file('./Input_Data/HIFLD/Local_Law_Enforcement_Locations.geojson')
    local_eoc = gpd.read_file('./Input_Data/HIFLD/Local_Emergency_Operations_Centers_EOC.geojson')
    state_eoc = gpd.read_file('./Input_Data/HIFLD/State_Emergency_Operations_Centers_EOC.geojson')

    # Convert Coordinates
    fire = fire.to_crs(epsg=crs_main)
    police = police.to_crs(epsg=crs_main)
    local_eoc = local_eoc.to_crs(epsg=crs_main)
    state_eoc = state_eoc.to_crs(epsg=crs_main)

    # Assign Census Block and Tract to HIFLD Data
    fire = pre.assign_census_hifld(fire, hayward_blocks, hayward_tracts, 'GEOID20') # MTL CHANGE NSI26 new input
    police = pre.assign_census_hifld(police, hayward_blocks, hayward_tracts, 'GEOID20') # MTL CHANGE NSI26 new input
    local_eoc = pre.assign_census_hifld(local_eoc, hayward_blocks, hayward_tracts, 'GEOID20') # MTL CHANGE NSI26 new input
    state_eoc = pre.assign_census_hifld(state_eoc, hayward_blocks, hayward_tracts, 'GEOID20') # MTL CHANGE NSI26 new input


    ##### MERGE DATA FOR FUTURE USE #####

    # Reduce columns and set class
    fire = fire[['geometry', 'CensusBlock', 'CensusTract']]
    fire['NSI_OccupancyClass'] = "GOV2-FIRE"
    police = police[['geometry', 'CensusBlock', 'CensusTract']]
    police['NSI_OccupancyClass'] = "GOV2-POLICE"
    local_eoc = local_eoc[['geometry', 'CensusBlock', 'CensusTract']]
    local_eoc['NSI_OccupancyClass'] = "GOV2-OPERATIONS"
    state_eoc = state_eoc[['geometry', 'CensusBlock', 'CensusTract']]
    state_eoc['NSI_OccupancyClass'] = "GOV2-OPERATIONS"

    # Concatinate and prepare for merge
    gov2_import = pd.concat([fire, police, local_eoc, state_eoc], ignore_index=True, sort=False)


    ##### SYNTHESIZE NSI GOV2 AND HIFLD POINTS #####
    # nsi = fxns.json_to_gdf(dir_intermediate + 'NSI_EDU1_EDU2_Upgrade.json', crs_main)

    # Set flag to true to display HIFLD and NSI GOV2 Points
    nsi = pre.synthesize_gov2_and_HIFLD_nsi26_update(nsi, gov2_import, drop_unpaired_nsi_gov2, drop_gov1_near_gov2, gov2_far_from_hifld)


In [ ]:
#### CREATE ID AND SET COLUMNS #####
nsi = pre.add_nsi_tracking_columns(nsi, nsi_filter_limit)

### COMPUTE MIN AND MAX OF RANGE OF RES UNITS BASED ON OCCUPANCY CLASS ### 
nsi = pre.compute_min_mix_units(nsi)



### DEFINE COLUMNS TO BE SUMMED OR COVERTED TO LISTS WHEN COMBINED ### 
# sum_columns get summed when points are merged (i.e. replacement cost)
# list_columns get put into lists (if information differs) when points are merged (i.e. foundation type)

# Set category of certain column names
excluded = ['geometry', 'CensusBlock', 'CensusTract', 'POINT_ID', 'NSI_OccupancyClass', 'POINT_DropFlag', 'POINT_DropNote', 'NSI_OC_Update',
            'POINT_FootprintID', 'DistanceToFtpt', 'ClosestFtpt_ID', 'POINT_MergeFlag','POINT_DataUpdate']
sum_columns = ['NSI_PopOver65_Day', 'NSI_PopUnder65_Day', 'NSI_Population_Day',
                    'NSI_PopOver65_Night', 'NSI_PopUnder65_Night', 'NSI_Population_Night','NSI_ContentValue','NSI_ReplacementCost',
                    'NSI_StructureValue','NSI_MinResUnits', 'NSI_MaxResUnits','POINT_NumPoints','NSI_Units','NSI_Students']
list_columns = ['NSI_FoundationType','NSI_FoundationHeight','NSI_BuildingType','NSI_MedYearBuilt',
                'NSI_NumberOfStories', 'POINT_Source', 'NSI_OrigSource', 'NSI_OrigFtptSource','NSI_BID','POINT_ID_List','NSI_TotalAreaSqFt']

# Check that all columns are assigned to a category
fxns.check_column_assignment(nsi, sum_columns, list_columns, excluded)


##### Group NSI rows with same BID ####

##### SPLIT DATA BASED ON DROP_FLAG #####
nsi_length = len(nsi) # Used for tracking purposes 
nsi0 = nsi[nsi['POINT_DropFlag']!=1] # Not dropped - these should be used in merge
nsi1 = nsi[nsi['POINT_DropFlag']==1] # Dropped - these should not be used in merge 

# Assuming values with same BID are all intended to be in the same footprint moving forward 
nsi0 = pre.merge_duplicate_bid(nsi0, list_columns, sum_columns)  

##### REUNITE DATA BASED ON DROP_FLAG AND CHECK THAT ALL POINTS WERE RETAINED #####
nsi = pre.recombine_dropped_data(nsi0, nsi1, nsi_length)

##### SAVE NSI #####
fxns.gdf_to_json(nsi, directory + 'NSI_for_Merge.json')

    

## ATTRIBUTE POINTS TO FOOTPRINTS

In [ ]:

# Target Directory 
directory = f'./Inventory_Outputs/{name}/'

# Make directory and intermediate directory
dir_attribution = directory + 'FootprintAttribution/'
dir_generation = directory + 'InventoryGeneration/'
dir_r2d = './R2D_Analysis/Inventories/' + name + '/'

os.makedirs(directory, exist_ok=True)
os.makedirs(dir_attribution, exist_ok=True)
os.makedirs(dir_generation, exist_ok=True)
os.makedirs(dir_r2d, exist_ok=True)


In [ ]:
##### LOAD PREPROCESSED DATA FOR MERGE #####
points = fxns.json_to_gdf('./Input_Data/ProcessedData/National/NSI_for_Merge.json', crs_main)

##### DISPLAY NUMBER OF POINTS #####
points_length = len(points) # Used for tracking purposes 
print('NSI:', len(points))
print('Footprints:', len(footprints))


In [ ]:

##### SPLIT DATA BASED ON DROP_FLAG #####
points0 = points[points['POINT_DropFlag']!=1] # Not dropped - these should be used in merge
points1 = points[points['POINT_DropFlag']==1] # Dropped - these should not be used in merge 


##### RUN FUNCTION TO MERGE CASES WITH ONE POINT WITHIN ONE FOOTPRINT #####
points0, map = pt_ftpt.merge_intersecting(points0, footprints, crs_plot, plot=False)

# Update MergeFlag99 for footprints that are larger than their designated occupancy type 
points0 = pt_ftpt.update_mergeflag99(points0, footprints, mergeflag = 1)

##### REUNITE DATA BASED ON DROP_FLAG AND CHECK THAT ALL POINTS WERE RETAINED #####
points = pt_ftpt.recombine_dropped_data(points0, points1, points_length)

##### CHECK FOR DUPLICATED FOOTPRINTS IN MERGED DATA #####
pt_ftpt.check_post_merge_duplicates(points.copy())

##### SAVE JSON FILE #####
# fxns.gdf_to_json(points.copy(), dir_intermediate + 'MergeFlag1.json')




##### LOAD DATA #####
# points = fxns.json_to_gdf(dir_intermediate + 'MergeFlag1.json', crs_main)


##################################################################################################################################

############# THE FOLLOWING CAN BE FILLED OUT TO MAKE MODIFICATIONS BASED ON THE PRINTED OUT "ODD OCCUPANCY PAIRINGS" #################

##### MANUALLY ASSIGN OCCUPANCY CLASS FOR FOOTPRINTS HERE #####
manually_assigned_occupancy_data = {
    "FootprintID":[], # Put FootprintID values in this list
    "POINT_OccupancyClass":[]} # Put corresponding occupancy class values in this list. Can enter string or list, 
                            # i.e. "NSI_OccupancyClass":['RES1',['IND3','RES3C]] would correspond to first and second FootprintID entered in list above
manually_assigned_occupancy = pd.DataFrame(manually_assigned_occupancy_data)

##### MANUALLY DROP NSI POINTS HERE #####
ids_to_drop = [] # Place NSI_ID values here that should be dropped from the merge 
points = pt_ftpt.drop_ids(points, ids_to_drop, 'Manually dropped due to occupancy class incompatibility')

##################################################################################################################################


##### SPLIT DATA BASED ON DROP_FLAG #####
points0 = points[points['POINT_DropFlag']!=1] # Not dropped - these should be used in merge
points1 = points[points['POINT_DropFlag']==1] # Dropped - these should not be used in merge 


#### RUN FUNCTION TO MERGE CASES WITH MULTIPLE POINTS WITHIN ONE FOOTPRINT #####
# Set flag to print odd occupancy pairings, including (RES + IND), (RES + GOV), and (EDU + IND) - does not change function outputs, only displays 
print_odd_occupancy_pairings = False
use_size_limit = True 
use_nsi_occupancy_merge = True # Set to true if points df consists of NSI data
points0 = pt_ftpt.address_overlapping_points(points0.copy(), footprints.copy(), list_columns, sum_columns, manually_assigned_occupancy,use_size_limit, use_nsi_occupancy_merge, print_odd_occupancy_pairings, crs_plot)

##### REUNITE DATA BASED ON DROP_FLAG AND CHECK THAT ALL POINTS WERE RETAINED #####
points = pt_ftpt.recombine_dropped_data(points0, points1, points_length)

##### CHECK FOR DUPLICATED FOOTPRINTS IN MERGED DATA #####
points['POINT_FootprintID'] = points['POINT_FootprintID'].astype('Int64')
pt_ftpt.check_post_merge_duplicates(points.copy())

##### SAVE JSON FILE #####
# fxns.gdf_to_json(points.copy(), dir_intermediate + 'MergeFlag2.json')








if drop_gov1_ind4_ind5: 
    ##### LOAD DATA #####
    # points = fxns.json_to_gdf(dir_intermediate + 'MergeFlag2.json', crs_main)

    ##### SPLIT DATA BASED ON DROP_FLAG #####
    points0 = points[points['POINT_DropFlag']!=1] # Not dropped - these should be used in merge
    points1 = points[points['POINT_DropFlag']==1] # Dropped - these should not be used in merge 

    ##### COLLECT REMAINING GOV1, IND4, and IND5 POINTS #####
    remaining_points, remaining_ftpt = pt_ftpt.find_remaining(points0, footprints, 'POINT_FootprintID','POINT_MergeFlag')
    points_to_drop = remaining_points[((remaining_points['NSI_OccupancyClass']=='IND4') | (remaining_points['NSI_OccupancyClass']=='IND5') | (remaining_points['NSI_OccupancyClass']=='GOV1'))]
    ids_to_drop = [item for sublist in points_to_drop['POINT_ID'] for item in (sublist if isinstance(sublist, list) else [sublist])]
    print(len(ids_to_drop), 'points to drop')


    ### DROP POINTS ###
    points0 = pt_ftpt.drop_ids(points0, ids_to_drop, 'IND4, IND5, or GOV1 point outside of footprint')

    ##### REUNITE DATA BASED ON DROP_FLAG AND CHECK THAT ALL POINTS WERE RETAINED #####
    points = pt_ftpt.recombine_dropped_data(points0, points1, points_length)

    ##### CHECK FOR DUPLICATED FOOTPRINTS IN MERGED DATA #####
    points['POINT_FootprintID'] = points['POINT_FootprintID'].astype('Int64')
    pt_ftpt.check_post_merge_duplicates(points.copy())

    ##### SAVE JSON FILE #####
    # fxns.gdf_to_json(points.copy(), dir_intermediate + 'MergeFlag2_dropped.json')











# Load data 
# points = fxns.json_to_gdf(dir_intermediate + '/MergeFlag2_dropped.json', crs_main)


##################################################################################################################################

############# THE FOLLOWING CAN BE FILLED OUT TO MAKE MODIFICATIONS BASED ON THE PRINTED OUT "ODD OCCUPANCY PAIRINGS" #################

##### MANUALLY ASSIGN OCCUPANCY CLASS FOR FOOTPRINTS HERE #####
manually_assigned_occupancy_data = {
    "FootprintID":[], # Put FootprintID values in this list
    "POINT_OccupancyClass":[]} # Put corresponding occupancy class values in this list. Can enter string or list, 
                            # i.e. "NSI_OccupancyClass":['RES1',['IND3','RES3C]] would correspond to first and second FootprintID entered in list above
manually_assigned_occupancy = pd.DataFrame(manually_assigned_occupancy_data)

##### MANUALLY DROP NSI POINTS HERE #####
ids_to_drop = [] # Place POINT_ID values here that should be dropped from the merge 
points = pt_ftpt.drop_ids(points, ids_to_drop, 'Manually dropped due to occupancy class incompatibility')

##################################################################################################################################


##### SPLIT DATA BASED ON DROP_FLAG #####
points0 = points[points['POINT_DropFlag']!=1].copy() # Not dropped - these should be used in merge
points1 = points[points['POINT_DropFlag']==1].copy() # Dropped - these should not be used in merge 


##### CONDUCT MERGE #####

# Get list of census blocks
bounding_id_list =  hayward_blocks['CensusBlock'].unique()
bounding_id_name = 'CensusBlock'
bounding_geometry = hayward_blocks.copy()

# Merge data using function 

points0 = pt_ftpt.distance_limit_merge(bounding_id_list, points0.copy(), footprints, bounding_id_name, manually_assigned_occupancy, list_columns, sum_columns, bounding_geometry, crs_plot,
                            distance_limit = 10, # Meters
                            use_surrounding_bgs = use_surrounding_bgs, # Footprints in surrounding bounding geometries (parcels) will be considered for each NSI point
                            prioritize_empty_footprints = True, # Empty footprints within the distance limit will be prioritized over partially full or full footprints for attribution 
                            prioritize_partial_footprints = prioritize_partial_footprints, # Footprints with MergeFlag = 99 will be prioritized over full footprints for attribution (prioritized second to empty footprints if prioritize_empty_footprints = True)
                            use_full_footprints = use_full_ftpt, # Full footprints will be considered for atribution (once empty and partial footprints in distance limit have been exhausted, if the above prioritize flags are set to True)
                            merge_flag = 310, 
                            use_size_limit = True, # Use size limit to designate "partially full footprints"
                            use_nsi_occupancy_merge = True, # Set to True only if merging NSI data 
                            print_odd_occupancy_pairings = False) # If True, some occupancy class pairs will be printed out if they are merged into same footprint to manually check 


##### REUNITE DATA BASED ON DROP_FLAG AND CHECK THAT ALL POINTS WERE RETAINED #####
points = pt_ftpt.recombine_dropped_data(points0, points1, points_length)

##### CHECK FOR DUPLICATED FOOTPRINTS IN MERGED DATA #####
points['POINT_FootprintID'] = points['POINT_FootprintID'].astype('Int64')
pt_ftpt.check_post_merge_duplicates(points.copy())

##### SAVE JSON FILE #####
# fxns.gdf_to_json(points.copy(), dir_intermediate + 'MergeFlag310.json')





if final_distance_bound > 10: 

    # Load data 
    # points = fxns.json_to_gdf(dir_intermediate + '/MergeFlag310.json', crs_main)

    ##################################################################################################################################

    ############# THE FOLLOWING CAN BE FILLED OUT TO MAKE MODIFICATIONS BASED ON THE PRINTED OUT "ODD OCCUPANCY PAIRINGS" #################

    ##### MANUALLY ASSIGN OCCUPANCY CLASS FOR FOOTPRINTS HERE #####
    manually_assigned_occupancy_data = {
        "FootprintID":[], # Put FootprintID values in this list
        "POINT_OccupancyClass":[]} # Put corresponding occupancy class values in this list. Can enter string or list, 
                                # i.e. "NSI_OccupancyClass":['RES1',['IND3','RES3C]] would correspond to first and second FootprintID entered in list above
    manually_assigned_occupancy = pd.DataFrame(manually_assigned_occupancy_data)

    ##### MANUALLY DROP NSI POINTS HERE #####
    ids_to_drop = [] # Place POINT_ID values here that should be dropped from the merge 
    points = pt_ftpt.drop_ids(points, ids_to_drop, 'Manually dropped due to occupancy class incompatibility')

    ##################################################################################################################################

    ##### SPLIT DATA BASED ON DROP_FLAG #####
    points0 = points[points['POINT_DropFlag']!=1].copy() # Not dropped - these should be used in merge
    points1 = points[points['POINT_DropFlag']==1].copy() # Dropped - these should not be used in merge 

    ##### CONDUCT MERGE #####

    # Get list of census blocks
    bounding_id_list =  hayward_blocks['CensusBlock'].unique()
    bounding_id_name = 'CensusBlock'
    bounding_geometry = hayward_blocks.copy()

    # Merge data using function 

    points0 = pt_ftpt.distance_limit_merge(bounding_id_list, points0.copy(), footprints, bounding_id_name, manually_assigned_occupancy, list_columns, sum_columns, bounding_geometry, crs_plot,
                                distance_limit = final_distance_bound, # Meters
                                use_surrounding_bgs = use_surrounding_bgs, # Footprints in surrounding bounding geometries (parcels) will be considered for each NSI point
                                prioritize_empty_footprints = True, # Empty footprints within the distance limit will be prioritized over partially full or full footprints for attribution 
                                prioritize_partial_footprints = prioritize_partial_footprints, # Footprints with MergeFlag = 99 will be prioritized over full footprints for attribution (prioritized second to empty footprints if prioritize_empty_footprints = True)
                                use_full_footprints = use_full_ftpt, # Full footprints will be considered for atribution (once empty and partial footprints in distance limit have been exhausted, if the above prioritize flags are set to True)
                                merge_flag = 3100, 
                                use_size_limit = True, # Use size limit to designate "partially full footprints"
                                use_nsi_occupancy_merge = True, # Set to True only if merging NSI data 
                                print_odd_occupancy_pairings = False) # If True, some occupancy class pairs will be printed out if they are merged into same footprint to manually check 


    ##### REUNITE DATA BASED ON DROP_FLAG AND CHECK THAT ALL POINTS WERE RETAINED #####
    points = pt_ftpt.recombine_dropped_data(points0, points1, points_length)

    ##### CHECK FOR DUPLICATED FOOTPRINTS IN MERGED DATA #####
    points['POINT_FootprintID'] = points['POINT_FootprintID'].astype('Int64')
    pt_ftpt.check_post_merge_duplicates(points.copy())

    ##### SAVE JSON FILE #####
    # fxns.gdf_to_json(points.copy(), dir_intermediate + 'MergeFlag3100.json')





##### DROP REMAINING NSI POINTS #####


# Load Dataset
# points = fxns.json_to_gdf(dir_intermediate + 'MergeFlag3100.json', crs_main)
points_length = len(points)

##### SPLIT DATA BASED ON DROP_FLAG #####
points0 = points[points['POINT_DropFlag']!=1].copy() # Not dropped - these should be used in merge
points1 = points[points['POINT_DropFlag']==1].copy() # Dropped - these should not be used in merge 

# Find remaining points without footprints and flag them as dropped
points0['POINT_FootprintID'] = points0['POINT_FootprintID'].apply(lambda x: int(x) if pd.notna(x) else x)
remaining_points, remaining_ftpt = pt_ftpt.find_remaining(points0, footprints,'POINT_FootprintID','POINT_MergeFlag')
points0 = pt_ftpt.drop_ids(points0, remaining_points['POINT_ID'].values, 'Points remaining after MergeFlag 3100')

##### REUNITE DATA BASED ON DROP_FLAG AND CHECK THAT ALL POINTS WERE RETAINED #####
points = pt_ftpt.recombine_dropped_data(points0, points1, points_length)

##### CHECK FOR DUPLICATED FOOTPRINTS IN MERGED DATA #####
points['POINT_FootprintID'] = points['POINT_FootprintID'].astype('Int64')
pt_ftpt.check_post_merge_duplicates(points.copy())

# ##### SAVE JSON FILE #####
# fxns.gdf_to_json(points.copy(), dir_intermediate + 'National_MergedPoints.json')




### CONVERT TO FOOTPRINTS 
# # Load point data 
# points = fxns.json_to_gdf(dir_intermediate + 'National_MergedPoints.json', crs_main)
points0 = points[points['POINT_DropFlag']!=1].copy() # Not dropped - these should be used in merge
points1 = points[points['POINT_DropFlag']==1].copy() # Dropped - these should not be used in merge 

# Drop geometry information in preparation for merge 
points0 = points0.drop(columns = ['geometry'])

# Drop additional columns used for tracking purposes 
points0 = points0.drop(columns = ['CensusTract','CensusBlock','POINT_DropFlag','POINT_DropNote','DistanceToFtpt', 'ClosestFtpt_ID','POINT_ID', 'NSI_OccupancyClass', 'POINT_DataUpdate'])
points0 = points0.rename(columns={'NSI_OC_Update': 'NSI_OccupancyClass'})
points0 = points0.rename(columns={'POINT_ID_List': 'POINT_ID'})

# Convert numeric, single-entry columns to appropriate type 
float_columns = ['NSI_ContentValue', 'NSI_StructureValue', 'NSI_ReplacementCost','NSI_PopOver65_Night',
                'NSI_PopUnder65_Night', 'NSI_Population_Night', 'NSI_PopOver65_Day', 'NSI_PopUnder65_Day', 'NSI_Population_Day']
int_columns = ['POINT_NumPoints', 'POINT_FootprintID', 'POINT_MergeFlag', 'NSI_MinResUnits', 'NSI_MaxResUnits']   


# Convert columns to appropriate type 
for col in float_columns:
    points0[col] = points0[col].astype(float)
for col in int_columns:
    points0[col] = points0[col].astype(int)
points0['NSI_OccupancyClass'] = points0['NSI_OccupancyClass'].apply(pt_ftpt.convert_to_list)
points0['POINT_ID'] = points0['POINT_ID'].apply(pt_ftpt.convert_to_list)






# Create baseline inventory 
ftpt_inv = footprints.copy()
orig_inv_length = len(ftpt_inv)
points0['National_Flag'] = 1

# Merge unique footprint information in with baseline footprint inventory and perform checks for dropped points
ftpt_inv = ftpt_inv.merge(points0, left_on='FootprintID', right_on='POINT_FootprintID', how='left')
if len(ftpt_inv) != orig_inv_length: 
    raise ValueError('Footprints Dropped - Step 1')

if len(ftpt_inv[ftpt_inv['National_Flag']==1]) != len(points0): 
    raise ValueError('Footprints Dropped - Step 2')

# Drop redundant footprint column
ftpt_inv = ftpt_inv.drop(columns = ['POINT_FootprintID'])
ftpt_inv['National_Flag'] = ftpt_inv['National_Flag'].fillna(0)

# Modify footprint inventory geometry to be the centroid of each footprint 
ftpt_inv = ftpt_inv.rename(columns={'geometry': 'ftpt_geometry'})
ftpt_inv['geometry'] = ftpt_inv['ftpt_geometry'].centroid
ftpt_inv.set_geometry('geometry')
ftpt_inv['Footprint_Flag'] = 1

# Save inventory 
fxns.gdf_to_json(ftpt_inv, dir_attribution + 'National_Inventory_Point.json')

## GENERATE INVENTORY

In [ ]:
# Load Inventory
inv_raw = fxns.json_to_gdf(dir_attribution + 'National_Inventory_Point.json', crs_main)
print(len(inv_raw))

# Create modifiable instance of raw inventory 
inv_mod = inv_raw.copy()

In [ ]:
## RESOLVE WITHIN-SOURCE DISAGREEMENT (MINIMAL BETWEEN SOURCE DISAGREEMENT DUE TO USE OF ONlY NATIONAL DATASETS)

# Columns that are modified to a single value (down from a list) by selecting mode 
modified_to_single_solo = ['NSI_BuildingType', 'NSI_MedYearBuilt', 'NSI_NumberOfStories', 'NSI_TotalAreaSqFt']
for col in modified_to_single_solo:
    inv_mod[col + '_Single'] = inv_mod[col].apply(resolve.modify_to_single_val)

# Assign paired selections for foundation type and height 
# These variables, since they are directly related, will always be taken from the same original data point
inv_mod[['NSI_FoundationType_Single', 'NSI_FoundationHeight_Single']] = inv_mod[['NSI_FoundationType', 'NSI_FoundationHeight']].apply(resolve.modify_to_single_val_paired, axis=1).apply(pd.Series)

# Update occupancy class based on rulesets specific for NSI/HIFLD Occupancy Class 
inv_mod[['NSI_OccupancyClass_Single','NSI_OccupancyClass_MixedUse']] = inv_mod['NSI_OccupancyClass'].apply(resolve.modify_to_single_nsi_occupancy).apply(pd.Series)
inv_mod['NSI_OccupancyClass_Single'] = inv_mod['NSI_OccupancyClass_Single'].fillna('')

# Simplify NSI Occupancies Containing RES1 to RES1 
inv_mod.loc[inv_mod['NSI_OccupancyClass_Single'].str.contains('RES1', na=False), 'NSI_OccupancyClass_Single'] = 'RES1'



## SAMPLE YEAR IF REQUESTED
if sample_year:
    inv_mod['Year_Delta'] = np.random.normal(loc=0, scale=20, size=len(inv_mod))
    inv_mod['NSI_MedYearBuilt_Single'] = inv_mod['NSI_MedYearBuilt_Single'] + round(inv_mod['Year_Delta'])


### MOBILE HOME POLYGONS 
if include_manual_mh_polygons: 

    ### USE MANUALLY GENERATED POLYGONS TO FORCE RES2 OCCUPANCY
    mh_polygons = gpd.read_file('./Input_Data/MH_Manual/MH_Hayward_Manual_Polygons.csv')
    mh_polygons.set_crs(crs_plot, inplace = True)
    mh_polygons.to_crs(crs_main, inplace = True)

    # Find inventory points that are within RES2 polygons and reset 
    # This converts footprints, even that do not currently contain NSI data. Flag is updated to reflect there is now data from a national source 
    in_polygons = gpd.sjoin(inv_mod, mh_polygons)
    inv_mod.loc[in_polygons.index, 'NSI_OccupancyClass_Single'] = 'RES2'
    inv_mod.loc[in_polygons.index, 'National_Flag'] = 1

    # Since all data is already integrated into NSI_OccupancyClass_Single, set to be equal to OccupancyClass_Best
    inv_mod['OccupancyClass_Best'] = inv_mod['NSI_OccupancyClass_Single']

    # Modify so RES2 are all single unit, single story structures (assumed based on spot checking google maps) 
    inv_mod.loc[inv_mod['OccupancyClass_Best'].str.contains('RES2', na=False), 'NSI_MinResUnits'] = 1
    inv_mod.loc[inv_mod['OccupancyClass_Best'].str.contains('RES2', na=False), 'NSI_MaxResUnits'] = 1
    inv_mod.loc[inv_mod['OccupancyClass_Best'].str.contains('RES2', na=False), 'NSI_NumberOfStories_Single'] = 1
    inv_mod.loc[inv_mod['OccupancyClass_Best'].str.contains('RES2', na=False), 'NSI_BuildingType_Single'] ='H'
    
else: 
    # Since all data is already integrated into NSI_OccupancyClass_Single, set to be equal to OccupancyClass_Best
    inv_mod['OccupancyClass_Best'] = inv_mod['NSI_OccupancyClass_Single']


## LINK TO 2020 CENSUS BLOCKS TO ENABLE UNIT SCALING BASED ON DECENNIAL CENSUS DATA 

# Load footprint-based inventory and 2020 Census Blocks 
inventory = inv_mod.copy()
inv_length = len(inventory)

# Find corresponding 2020 Census Block 
inventory = inventory.sjoin(hayward_blocks[['GEOID20','geometry']], how='left')
inventory = inventory.rename(columns = {'GEOID20':'CensusBlock_2020'})

# Filter inventory for footprints with NSI points 
inventory0 = inventory[inventory['National_Flag']==0]
inventory = inventory[inventory['National_Flag']==1]

# Filter inventory for footprints that are outside vs inside 2020 census blocks 
inventory_outside2020blocks = inventory[inventory['CensusBlock_2020'].isna()].copy()

# Assign footprints outside of census block to nearest census block 
inventory_outside2020blocks['Nearest'] = inventory_outside2020blocks['geometry'].apply(lambda point: resolve.outside_ftpt_nearest_cb(point, hayward_blocks.to_crs(crs_main)[['GEOID20','geometry']]))
inventory.loc[inventory_outside2020blocks.index, 'CensusBlock_2020'] = inventory_outside2020blocks['Nearest'].values

### DOWNLOAD POPULATION AND NUMBER OF UNITS FOR 2020 CENSUS BLOCKS
cbs = resolve.download_census_data(census_api_key, hayward_blocks, state_fips, county_fips)
# cbs = pd.read_csv('./Input_Data/Census/2020_Census_Units.csv')


### ESTIMATE THE NUMBER OF UNITS USING CENSUS INFORMATION AND WITH POPULATION SCALING ###
# Pre-set modification flag to be 0
inventory['Flag_ModifiedByCensus'] = 0
inventory0['Flag_ModifiedByCensus'] = 0

# Assign number of units using information from census block 
inventory2 = resolve.assign_units_from_censusblock(inventory.copy(), 'CensusBlock_2020', cbs)

# Re-combine updated inventory with footprints with no NSI points 
inventory = resolve.recombine_dropped_data(inventory2, inventory0, inv_length)

# Drop appropriate columns 
inventory = inventory.drop(columns = ['index_right','CensusBlock_2020'])
inv_mod = inventory.copy()





if unit_source == 'census':

    # Assign best value for number of units
    inv_mod['Units_Best'] = inv_mod.apply(
        lambda row: (row['Units_CensusEstimate'] if pd.notna(row['Units_CensusEstimate'])
        else (row['NSI_MinResUnits'] if pd.notna(row['NSI_MinResUnits']) 
        else np.nan)), axis=1)

    inv_mod['Units_Best_Source'] = inv_mod.apply(
        lambda row: ('Units_CensusEstimate' if pd.notna(row['Units_CensusEstimate'])
        else ('NSI_MinResUnits' if pd.notna(row['NSI_MinResUnits']) 
        else 'None')), axis=1)

    # Modulate RES3 Occupancy based on updated number of units 
    inv_mod['OccupancyClass_Best'] = inv_mod.apply(resolve.update_res_occ, axis=1)


elif unit_source == 'nsi_min':

    # Assign best value for number of units
    inv_mod['Units_Best'] = inv_mod.apply(
        lambda row: (row['NSI_MinResUnits'] if pd.notna(row['NSI_MinResUnits'])
        else (row['Units_CensusEstimate'] if pd.notna(row['Units_CensusEstimate']) 
        else np.nan)), axis=1)

    inv_mod['Units_Best_Source'] = inv_mod.apply(
        lambda row: ('NSI_MinResUnits' if pd.notna(row['NSI_MinResUnits'])
        else ('Units_CensusEstimate' if pd.notna(row['Units_CensusEstimate']) 
        else 'None')), axis=1)

    # Modulate RES3 Occupancy based on updated number of units 
    inv_mod['OccupancyClass_Best'] = inv_mod.apply(resolve.update_res_occ, axis=1)


elif unit_source == 'nsi_max':

    # Assign best value for number of units
    inv_mod['Units_Best'] = inv_mod.apply(
        lambda row: (row['NSI_MaxResUnits'] if pd.notna(row['NSI_MaxResUnits'])
        else (row['Units_CensusEstimate'] if pd.notna(row['Units_CensusEstimate']) 
        else np.nan)), axis=1)

    inv_mod['Units_Best_Source'] = inv_mod.apply(
        lambda row: ('NSI_MaxResUnits' if pd.notna(row['NSI_MaxResUnits'])
        else ('Units_CensusEstimate' if pd.notna(row['Units_CensusEstimate']) 
        else 'None')), axis=1)

    # Modulate RES3 Occupancy based on updated number of units 
    inv_mod['OccupancyClass_Best'] = inv_mod.apply(resolve.update_res_occ, axis=1)


elif unit_source == 'nsi_mean':

    inv_mod['NSI_MeanResUnits'] = round((inv_mod['NSI_MinResUnits'] + inv_mod['NSI_MaxResUnits'])/2)

    # Assign best value for number of units
    inv_mod['Units_Best'] = inv_mod.apply(
        lambda row: (row['NSI_MeanResUnits'] if pd.notna(row['NSI_MeanResUnits'])
        else (row['Units_CensusEstimate'] if pd.notna(row['Units_CensusEstimate']) 
        else np.nan)), axis=1)

    inv_mod['Units_Best_Source'] = inv_mod.apply(
        lambda row: ('NSI_MeanResUnits' if pd.notna(row['NSI_MeanResUnits'])
        else ('Units_CensusEstimate' if pd.notna(row['Units_CensusEstimate']) 
        else 'None')), axis=1)

    # Modulate RES3 Occupancy based on updated number of units 
    inv_mod['OccupancyClass_Best'] = inv_mod.apply(resolve.update_res_occ, axis=1)









## STORIES 
# In the NSI data, there are points with very large number of stories that are unrealistic for Haywrd. This box selects how best to deal with those footprints. 
# The majority of footprints are not adjusted; only those over a user defined threshold will be modified. 

if reset_very_high_stories == 'Mean_of_Occupancy_Class':
    inv_mod['Stories_Best'] = resolve.reset_very_high_stories_to_mean(inv_mod, stories_limit)

elif reset_very_high_stories == 'Scale_from_Units':
    inv_mod['Stories_Best'] = resolve.reset_very_high_stories_by_units(inv_mod, stories_limit)

elif reset_very_high_stories == 'No_Reset':
    inv_mod['Stories_Best'] = inv_mod['NSI_NumberOfStories_Single']

else: 
    raise ValueError('Please Select Valid entry for "reset_very_high_stories"')










# PLAN AREA 
if area_source == 'nsi':

    inv_mod['PlanArea_Best'] = inv_mod.apply(
        lambda row: ((row['NSI_TotalAreaSqFt_Single'] / row['Stories_Best']) if (pd.notna(row['NSI_TotalAreaSqFt_Single']) and pd.notna(row['Stories_Best']))
        else (row['FootprintArea'] if pd.notna(row['FootprintArea']) 
        else np.nan)), axis=1)

    inv_mod['PlanArea_Best_Source'] = inv_mod.apply(
        lambda row: 'NSI_TotalAreaSqFt_Over_Stories' if pd.notna(row['NSI_TotalAreaSqFt_Single']) and pd.notna(row['Stories_Best'])
        else 'FootprintArea' if pd.notna(row['FootprintArea']) 
        else 'None',
        axis=1)
    
elif area_source == 'footprint':

    inv_mod['PlanArea_Best'] = inv_mod.apply(
        lambda row: (row['FootprintArea'] if pd.notna(row['FootprintArea']) 
        else np.nan), axis=1)

    inv_mod['PlanArea_Best_Source'] = inv_mod.apply(
        lambda row: 'FootprintArea' if pd.notna(row['FootprintArea']) 
        else 'None',
        axis=1)







### COMPUTE REPLACEMENT COST USING HAZUS VALUES * SQUARE FOOTAGE 

# Load Hazus cost values 
hazus_conversion = pd.read_csv('./Input_Data/National/Hazus_Cost.csv')

# Compute hazus replacement cost (set scaling for contents to be true)
include_scaling_for_contents = True 
inv_mod = resolve.compute_hazus_replacement_cost(inv_mod.copy(), hazus_conversion, include_scaling_for_contents)


# Remove cases where NSI value is listed as 0
inv_mod.loc[inv_mod[inv_mod['NSI_ReplacementCost'] == 0].index, 'NSI_ReplacementCost'] = np.nan


if cost_source == 'nsi':
    # Prioritize NSI Replacement Cost, then Hazus-computed replacement cost
    inv_mod['ReplacementCost_Best'] = inv_mod.apply(
        lambda row: row['NSI_ReplacementCost'] if pd.notna(row['NSI_ReplacementCost']) 
        else (row['ReplacementCost_Hazus'] if pd.notna(row['ReplacementCost_Hazus']) 
        else (np.nan)), axis=1)

    inv_mod['ReplacementCost_Best_Source'] = inv_mod.apply(
        lambda row: 'NSI_ReplacementCost' if pd.notna(row['NSI_ReplacementCost']) 
        else ('Hazus' if pd.notna(row['ReplacementCost_Hazus']) 
        else ('None')), axis=1)

elif cost_source == 'hazus':
    # Prioritize Hazus-computed replacement cost, then NSI Replacement Cost
    inv_mod['ReplacementCost_Best'] = inv_mod.apply(
        lambda row: row['ReplacementCost_Hazus'] if pd.notna(row['ReplacementCost_Hazus']) 
        else (row['NSI_ReplacementCost'] if pd.notna(row['NSI_ReplacementCost']) 
        else (np.nan)), axis=1)

    inv_mod['ReplacementCost_Best_Source'] = inv_mod.apply(
        lambda row: 'Hazus' if pd.notna(row['ReplacementCost_Hazus']) 
        else ('NSI_ReplacementCost' if pd.notna(row['NSI_ReplacementCost']) 
        else ('None')), axis=1)








## STRUCTURE VALUE 

### COMPUTE REPLACEMENT COST USING HAZUS VALUES * SQUARE FOOTAGE 

# Load Hazus cost values 
hazus_conversion = pd.read_csv('./Input_Data/National/Hazus_Cost.csv')

# Drop previous Hazus value 
inv_mod = inv_mod.drop(columns='ReplacementCost_Hazus')

# Compute hazus structure value (set scaling for contents to be false)
include_scaling_for_contents = False 
inv_mod = resolve.compute_hazus_replacement_cost(inv_mod.copy(), hazus_conversion, include_scaling_for_contents)


if cost_source == 'nsi':
    # Prioritize NSI Replacement Cost, then Hazus-computed replacement cost
    inv_mod['StructureValue_Best'] = inv_mod.apply(
        lambda row: row['NSI_StructureValue'] if pd.notna(row['NSI_StructureValue']) 
        else (row['ReplacementCost_Hazus'] if pd.notna(row['ReplacementCost_Hazus']) 
        else (np.nan)), axis=1)

    inv_mod['StructureValue_Best_Source'] = inv_mod.apply(
        lambda row: 'NSI_StructureValue' if pd.notna(row['NSI_StructureValue']) 
        else ('Hazus' if pd.notna(row['ReplacementCost_Hazus']) 
        else ('None')), axis=1)

elif cost_source == 'hazus':
    # Prioritize Hazus-computed replacement cost, then NSI Replacement Cost
    inv_mod['StructureValue_Best'] = inv_mod.apply(
        lambda row: row['ReplacementCost_Hazus'] if pd.notna(row['ReplacementCost_Hazus']) 
        else (row['NSI_StructureValue'] if pd.notna(row['NSI_StructureValue']) 
        else (np.nan)), axis=1)

    inv_mod['StructureValue_Best_Source'] = inv_mod.apply(
        lambda row: 'Hazus' if pd.notna(row['ReplacementCost_Hazus']) 
        else ('NSI_StructureValue' if pd.notna(row['NSI_StructureValue']) 
        else ('None')), axis=1)

# Export Inventory
fxns.gdf_to_json(inv_mod, dir_generation + 'Inventory_AllFields.json')

In [ ]:
# Load national inventory 
# data = fxns.json_to_gdf(dir_generation + 'Inventory_AllFields.json', crs_main)
data = inv_mod.copy()

# Remove cases of footprints with no data 
data = data[data['National_Flag']==1]

# Convert to format of R2D - keep missing data (for imputation purposes)
for_imputation = data.copy().to_crs(crs_plot)
for_imputation['Longitude'] = for_imputation['geometry'].x
for_imputation['Latitude'] = for_imputation['geometry'].y

# Separate required columns for imputation 
for_imputation = for_imputation[['Latitude','Longitude','PlanArea_Best','Stories_Best','NSI_MedYearBuilt_Single','ReplacementCost_Best','StructureValue_Best','OccupancyClass_Best', 'NSI_BuildingType_Single','Units_Best','NSI_Population_Night','NSI_Population_Day','CensusBlock','CensusTract','FootprintID']]

# Standardize columns for imputation and R2D 
for_imputation = for_imputation.rename(columns={
    'PlanArea_Best' : 'PlanArea',
    'Stories_Best': 'NumberOfStories',
    'NSI_MedYearBuilt_Single': 'YearBuilt',
    'OccupancyClass_Best': 'OccupancyClass',
    'NSI_BuildingType_Single':'BuildingType',
    'ReplacementCost_Best':'ReplacementCost',
    'StructureValue_Best':'StructureValue',
    'NSI_Population_Night':'NightPopulation',
    'NSI_Population_Day':'DayPopulation',
    'Units_Best':'NumberOfUnits'})

# Convert None for imputation types 
for_imputation['BuildingType'] = for_imputation['BuildingType'].replace('None', np.nan)

# Add index
for_imputation.insert(0, 'index', range(len(for_imputation)))

# Export inventory 
for_imputation.to_csv(dir_generation + 'Inventory_Before_Imputation.csv', index = False)




### IMPUTE

# Specify data for imputation
file_path = dir_generation + 'Inventory_Before_Imputation.csv'

# create an Import to get the classes
importer = Importer()
knn_imputer_class = importer.get_class("KnnImputer")

# Load inventory 
inventory = AssetInventory()
inventory.read_from_csv(file_path,keep_existing=True, id_column='index') 


#### IMPUTE DATA USING BRAILS ####
imputer=knn_imputer_class(inventory,n_possible_worlds=1, exclude_features=['PlanArea','ReplacementCost','StructureValue','OccupancyClass','FootprintID','BuildingType'])
new_inventory = imputer.impute()


# Conver to pandas geodataframe
inv_geoj = new_inventory.get_geojson()
gdf = gpd.GeoDataFrame.from_features(inv_geoj["features"])

# Correct data type for population
gdf['NightPopulation'] = gdf['NightPopulation'].replace('',0)
gdf['DayPopulation'] = gdf['DayPopulation'].replace('',0)

# ## SAVE IMPUTED INVENTORY
# fxns.gdf_to_json(gdf, dir_generation + 'Inventory_IMPUTED.json')




## INFER STRUCTURE TYPE 

data = gdf.copy()

# SET VARIABLE NAMES
occ_key = 'OccupancyClass'
nstory_key = 'NumberOfStories'
year_key = 'YearBuilt'
strtype_key = 'StructureType'
bldgtype_key = 'BuildingType'
n_pw = 1

# CALL FUNCTION TO INFER STRUCTURE TYPE
bldg_properties_df = resolve.infer_structure_type(data.copy(), state, occ_key, nstory_key, year_key, bldgtype_key, strtype_key, n_pw, use_nsi_bldg_type, allow_mh_only_for_res2, no_urm, res3ab_to_res1_flag)
bldg_properties_df = resolve.resolve_structure_errors(bldg_properties_df.copy(), fill_in_error_structure_types)

# Create appropriate columns
r2d = bldg_properties_df.copy()
r2d['Longitude'] = r2d['geometry'].x
r2d['Latitude'] = r2d['geometry'].y
r2d = r2d[['Latitude','Longitude','PlanArea','NumberOfStories','YearBuilt','ReplacementCost','StructureValue','StructureType','BuildingType','OccupancyClass_clean','OccupancyClass', 'NumberOfUnits','NightPopulation','DayPopulation','CensusBlock','CensusTract','FootprintID','geometry']]

# Update building type based on structure type assignment 
r2d['BuildingType'] = r2d['StructureType'].apply(resolve.extract_bldg_type)

# Assign design level and height class (used in regional analysis)
r2d = resolve.find_design_level(r2d, 'StructureType', 'YearBuilt', 'DesignLevel')
r2d = resolve.find_height_class(r2d, 'StructureType', 'NumberOfStories', 'HeightClass')

# Add id
r2d.insert(0, 'id', range(len(r2d)))

# Rename occupancy class columns for R2D use
r2d = r2d.rename(columns={'OccupancyClass': 'OccupancyClass_Actual',
                          'OccupancyClass_clean': 'OccupancyClass'})

# Save inventory
r2d.to_csv(dir_r2d + f'R2D_Inventory.csv', index = False)
fxns.gdf_to_json(r2d, dir_r2d + f'R2D_Inventory.json')



In [ ]:
# include_manual_mh_polygons = True
# allow_mh_only_for_res2 = True 
# use_nsi_bldg_type = False
# RESULTS: 2273 RES2, 2273 MH


# include_manual_mh_polygons = True
# allow_mh_only_for_res2 = True 
# use_nsi_bldg_type = False
# RESULTS: 81 RES2, 81 MH


# include_manual_mh_polygons = False
# allow_mh_only_for_res2 = False 
# use_nsi_bldg_type = False
# RESULTS: 81 RES2, 128 MH


# include_manual_mh_polygons = False
# allow_mh_only_for_res2 = False 
# use_nsi_bldg_type = True
# RESULTS: 81 RES2, 132 MH


# include_manual_mh_polygons = True
# allow_mh_only_for_res2 = True 
# use_nsi_bldg_type = True
# RESULTS: 2273 RES2, 2273 MH